# Prices — electricity prices and techno-economic cost inputs

This notebook is designed to sit beside:

- `2.tech_lca_foreground.ipynb`
- `3.custom_grid.ipynb`
- `4.wind_power.ipynb`
- `1.dashboard_lca*.ipynb`

It imports `dashboard_config.py`, mirrors the same time-window controls, and writes optimisation-ready pricing files to `PRICE_OUTPUT_DIR`.

Main outputs:

1. half-hourly Elexon Market Index electricity prices aligned to the same `DATETIME` slices used by the grid/wind notebooks;
2. UK onshore wind CAPEX/OPEX assumptions from the DESNZ/Arup 2024 onshore wind cost report;
3. PEM/Alkaline/SOEC electrolyser CAPEX/OPEX assumptions from the NESO Green Hydrogen Data Portal where available, otherwise a transparent low/central/high proxy table;
4. optional battery cost placeholders for your optimisation model.

The notebook deliberately separates **time-series prices** from **scenario cost assumptions**:

- Elexon prices vary by settlement period;
- wind, electrolyser and battery CAPEX are scenario parameters, not half-hourly variables.


In [2]:
# All settings come from dashboard_config.py where possible.
# This notebook also has safe defaults so it can run before you paste the optional
# PRICE settings into dashboard_config.py.

from pathlib import Path
import json
import math
import time
from datetime import timedelta

import numpy as np
import pandas as pd
import requests

try:
    import dashboard_config as cfg
    from dashboard_config import *
    HAS_CONFIG = True
except Exception as exc:
    HAS_CONFIG = False
    cfg = None
    print("Could not import dashboard_config.py; using local defaults only.")
    print(repr(exc))

try:
    import lca_helpers as H
    HAS_LCA_HELPERS = True
except Exception:
    HAS_LCA_HELPERS = False

if HAS_CONFIG and hasattr(cfg, "print_dashboard"):
    cfg.print_dashboard()

print("\nPrice notebook")
print("--------------")
print("dashboard_config loaded:", HAS_CONFIG)
print("lca_helpers loaded:     ", HAS_LCA_HELPERS)


/opt/miniconda3/envs/brightway/lib/python3.11/site-packages/bw2calc/__init__.py:54: UserWarning: 
It seems like you have an ARM architecture, but haven't installed scikit-umfpack:

    https://pypi.org/project/scikit-umfpack/

Installing it could give you much faster calculations.

  warnings.warn(UMFPACK_WARNING)


Master Dashboard
----------------
Project:                 hydrogen-smr
Foreground DB:           hydrogen foreground
Run adaptive foreground: True
Run adaptive grid:       True
Run adaptive wind/grid:  True
Run adaptive prices:     True
Grid data source:        carbon_api | notebook: 3.1.custom_grid_carbon_intensity_api.ipynb
Run grid scenario LCA:   True
Run wind/grid LCA:       True
Run price data:          True
Grid method:             cheap | loss factor: 1.0316426921769999
Wind/grid method:        cheap
Selected grid techs:     ['PEM operation']
Wind/grid mode:          both (blended + switching) | electrolyser(s): ['AE operation']
Price output dir:        price_outputs
Elexon provider:         APXMIDP | fallback: N2EXMIDP
Cost case:               central | wind: central

Price notebook
--------------
dashboard_config loaded: True
lca_helpers loaded:      True


## Price notebook settings

These settings can be pasted into `dashboard_config.py`. Until then, this notebook uses `getattr(cfg, "SETTING", default)` so it will not break your existing setup.


In [3]:
# =============================================================================
# Price notebook settings — override these by defining them in dashboard_config.py
# =============================================================================

RUN_PRICE_DATA = getattr(cfg, "RUN_PRICE_DATA", True) if HAS_CONFIG else True
PRICE_OUTPUT_DIR = Path(getattr(cfg, "PRICE_OUTPUT_DIR", "price_outputs")) if HAS_CONFIG else Path("price_outputs")
PRICE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the same time-window logic as the grid/wind notebooks.
PRICE_TIME_MODE = getattr(cfg, "PRICE_TIME_MODE", getattr(cfg, "GRID_TIME_MODE", "range")) if HAS_CONFIG else "range"
PRICE_SINGLE_DATETIME = getattr(cfg, "PRICE_SINGLE_DATETIME", getattr(cfg, "GRID_SINGLE_DATETIME", "2025-01-01 00:00:00")) if HAS_CONFIG else "2025-01-01 00:00:00"
PRICE_RANGE_START = getattr(cfg, "PRICE_RANGE_START", getattr(cfg, "GRID_RANGE_START", "2025-01-01 00:00:00")) if HAS_CONFIG else "2025-01-01 00:00:00"
PRICE_RANGE_END = getattr(cfg, "PRICE_RANGE_END", getattr(cfg, "GRID_RANGE_END", "2025-01-07 23:30:00")) if HAS_CONFIG else "2025-01-07 23:30:00"
PRICE_GRID_CSV_PATH = getattr(cfg, "PRICE_GRID_CSV_PATH", getattr(cfg, "CSV_PATH", "df_fuel_ckan.csv")) if HAS_CONFIG else "df_fuel_ckan.csv"

# Elexon BMRS Market Index Data.
ELEXON_API_BASE = getattr(cfg, "ELEXON_API_BASE", "https://data.elexon.co.uk/bmrs/api/v1") if HAS_CONFIG else "https://data.elexon.co.uk/bmrs/api/v1"
ELEXON_MARKET_INDEX_PROVIDER = getattr(cfg, "ELEXON_MARKET_INDEX_PROVIDER", "APXMIDP") if HAS_CONFIG else "APXMIDP"
ELEXON_MARKET_INDEX_FALLBACK_PROVIDER = getattr(cfg, "ELEXON_MARKET_INDEX_FALLBACK_PROVIDER", "N2EXMIDP") if HAS_CONFIG else "N2EXMIDP"
ELEXON_CHUNK_DAYS = int(getattr(cfg, "ELEXON_CHUNK_DAYS", 7)) if HAS_CONFIG else 7
ELEXON_REQUEST_SLEEP_S = float(getattr(cfg, "ELEXON_REQUEST_SLEEP_S", 0.15)) if HAS_CONFIG else 0.15
ELEXON_TIMEOUT_S = int(getattr(cfg, "ELEXON_TIMEOUT_S", 60)) if HAS_CONFIG else 60

# Delivered-price uplift: keep zero for pure wholesale-price modelling. Increase
# for a sensitivity case including supplier/network/policy adders.
ELECTRICITY_DELIVERED_UPLIFT_GBP_PER_MWH = float(getattr(cfg, "ELECTRICITY_DELIVERED_UPLIFT_GBP_PER_MWH", 0.0)) if HAS_CONFIG else 0.0

# NESO electrolyser cost API resource.
NESO_API_BASE = getattr(cfg, "NESO_API_BASE", "https://api.neso.energy/api/3/action") if HAS_CONFIG else "https://api.neso.energy/api/3/action"
NESO_ELECTROLYSER_RESOURCE_ID = getattr(cfg, "NESO_ELECTROLYSER_RESOURCE_ID", "88cfa584-db79-4573-a0c0-c14086257f8f") if HAS_CONFIG else "88cfa584-db79-4573-a0c0-c14086257f8f"
NESO_LIMIT = int(getattr(cfg, "NESO_LIMIT", 5000)) if HAS_CONFIG else 5000


# Electrolyser cost proxy fallback. NESO remains first choice, but this table keeps
# the optimisation/LCOH workflow runnable when the NESO resource is unavailable.
USE_ELECTROLYSER_PROXY_IF_NESO_EMPTY = bool(getattr(cfg, "USE_ELECTROLYSER_PROXY_IF_NESO_EMPTY", True)) if HAS_CONFIG else True
PROXY_ELECTROLYSER_BUILD_YEAR = int(getattr(cfg, "PROXY_ELECTROLYSER_BUILD_YEAR", 2025)) if HAS_CONFIG else 2025

PROXY_ELECTROLYSER_CAPEX_GBP_PER_KWE = getattr(cfg, "PROXY_ELECTROLYSER_CAPEX_GBP_PER_KWE", {
    "Alkaline": {"low": 700, "central": 1200, "high": 1700},
    "PEM":      {"low": 850, "central": 1450, "high": 2100},
    "SOEC":     {"low": 1500, "central": 2500, "high": 3500},
}) if HAS_CONFIG else {
    "Alkaline": {"low": 700, "central": 1200, "high": 1700},
    "PEM":      {"low": 850, "central": 1450, "high": 2100},
    "SOEC":     {"low": 1500, "central": 2500, "high": 3500},
}

PROXY_ELECTROLYSER_EFFICIENCY_PCT = getattr(cfg, "PROXY_ELECTROLYSER_EFFICIENCY_PCT", {
    "Alkaline": 70.0,
    "PEM": 67.0,
    "SOEC": 80.0,
}) if HAS_CONFIG else {"Alkaline": 70.0, "PEM": 67.0, "SOEC": 80.0}

PROXY_ELECTROLYSER_FIXED_OPEX_PCT_CAPEX = getattr(cfg, "PROXY_ELECTROLYSER_FIXED_OPEX_PCT_CAPEX", {
    "Alkaline": 0.03,
    "PEM": 0.04,
    "SOEC": 0.04,
}) if HAS_CONFIG else {"Alkaline": 0.03, "PEM": 0.04, "SOEC": 0.04}

PROXY_ELECTROLYSER_LIFETIME_YEARS = getattr(cfg, "PROXY_ELECTROLYSER_LIFETIME_YEARS", {
    "Alkaline": 20,
    "PEM": 20,
    "SOEC": 20,
}) if HAS_CONFIG else {"Alkaline": 20, "PEM": 20, "SOEC": 20}

# Which cost case is treated as the central optimisation case.
COST_CASE = getattr(cfg, "COST_CASE", "central") if HAS_CONFIG else "central"
WIND_COST_CASE = getattr(cfg, "WIND_COST_CASE", COST_CASE) if HAS_CONFIG else COST_CASE

# Optional battery placeholders. Keep these as None until you choose a source;
# the notebook writes the rows, but will not silently invent a battery cost.
BATTERY_CAPEX_GBP_PER_MWH = getattr(cfg, "BATTERY_CAPEX_GBP_PER_MWH", {"low": None, "central": None, "high": None}) if HAS_CONFIG else {"low": None, "central": None, "high": None}
BATTERY_FIXED_OPEX_PCT_CAPEX = getattr(cfg, "BATTERY_FIXED_OPEX_PCT_CAPEX", {"low": None, "central": None, "high": None}) if HAS_CONFIG else {"low": None, "central": None, "high": None}
BATTERY_LIFETIME_YEARS = getattr(cfg, "BATTERY_LIFETIME_YEARS", {"low": None, "central": None, "high": None}) if HAS_CONFIG else {"low": None, "central": None, "high": None}

if not RUN_PRICE_DATA:
    raise SystemExit("RUN_PRICE_DATA is False. Set it True in dashboard_config.py and rerun this notebook.")

print("PRICE_OUTPUT_DIR:", PRICE_OUTPUT_DIR.resolve())
print("PRICE_TIME_MODE: ", PRICE_TIME_MODE)
print("Elexon provider: ", ELEXON_MARKET_INDEX_PROVIDER)
print("Elexon fallback: ", ELEXON_MARKET_INDEX_FALLBACK_PROVIDER)
print("Delivered uplift:", ELECTRICITY_DELIVERED_UPLIFT_GBP_PER_MWH, "£/MWh")
print("NESO proxy fallback:", USE_ELECTROLYSER_PROXY_IF_NESO_EMPTY)


PRICE_OUTPUT_DIR: /Users/louis/brightway-hydrogen/split up version/price_outputs
PRICE_TIME_MODE:  range
Elexon provider:  APXMIDP
Elexon fallback:  N2EXMIDP
Delivered uplift: 0.0 £/MWh
NESO proxy fallback: True


## Select the same half-hourly slices as the grid/wind notebooks

Preferred route: use `lca_helpers.load_grid_csv()` and `lca_helpers.select_grid_rows()` so this notebook exactly mirrors the other notebooks.

Fallback route: if `lca_helpers.py` is not available, load the grid CSV directly and select `single` / `range` slices locally.


In [4]:
def _coerce_datetime_col(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with a normalised DATETIME column."""
    out = df.copy()
    if "DATETIME" not in out.columns:
        candidates = [c for c in out.columns if c.lower() in {"datetime", "timestamp", "time", "date"}]
        if not candidates:
            raise ValueError("No DATETIME-like column found in grid data.")
        out = out.rename(columns={candidates[0]: "DATETIME"})
    out["DATETIME"] = pd.to_datetime(out["DATETIME"], errors="coerce").dt.tz_localize(None)
    out = out.dropna(subset=["DATETIME"]).sort_values("DATETIME").reset_index(drop=True)
    return out


def load_price_run_rows() -> tuple[pd.DataFrame, str]:
    """Load the exact same timeslices used by the selected LCA grid workflow."""
    # Best route: when this notebook is called from 1.dashboard_lca_adaptive.ipynb,
    # 4 or 4.1 has already selected the canonical grid rows in the same kernel.
    if "DASHBOARD_GRID_RUN_ROWS" in globals():
        rows = _coerce_datetime_col(DASHBOARD_GRID_RUN_ROWS.copy())
        label = str(globals().get("DASHBOARD_GRID_TARGET_LABEL", "selected_grid_rows"))
        print("Using canonical grid rows from the adaptive dashboard run.")
        return rows, label

    selected_grid_source = getattr(cfg, "GRID_SOURCE_NORMALIZED", "csv") if HAS_CONFIG else "csv"

    # Direct-run fallback for the old CSV workflow.
    if HAS_LCA_HELPERS and selected_grid_source == "csv":
        grid_df = H.load_grid_csv()
        run_rows, target_label = H.select_grid_rows(grid_df)
        run_rows = _coerce_datetime_col(run_rows)
        return run_rows, str(target_label)

    # Local fallback if lca_helpers.py is not on the path or if explicitly using CSV.
    if selected_grid_source != "csv":
        raise RuntimeError(
            "PRICE notebook could not find DASHBOARD_GRID_RUN_ROWS for GRID_DATA_SOURCE='carbon_api'. "
            "Run 1.dashboard_lca_adaptive.ipynb with RUN_GRID_NOTEBOOK_FROM_DASHBOARD=True, "
            "or run 3.1.custom_grid_carbon_intensity_api.ipynb first in the same kernel."
        )

    csv_path = Path(PRICE_GRID_CSV_PATH)
    if not csv_path.exists() and Path("/mnt/data", PRICE_GRID_CSV_PATH).exists():
        csv_path = Path("/mnt/data", PRICE_GRID_CSV_PATH)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Could not find {PRICE_GRID_CSV_PATH}. Either run this in the project folder "
            "or make lca_helpers.py available."
        )
    df = _coerce_datetime_col(pd.read_csv(csv_path))

    mode = str(PRICE_TIME_MODE).lower()
    if mode == "single":
        t = pd.Timestamp(PRICE_SINGLE_DATETIME).tz_localize(None)
        idx = (df["DATETIME"] - t).abs().idxmin()
        rows = df.loc[[idx]].copy()
        label = t.strftime("%Y-%m-%dT%H-%M")
    elif mode in {"range", "year_average"}:
        start = pd.Timestamp(PRICE_RANGE_START).tz_localize(None)
        end = pd.Timestamp(PRICE_RANGE_END).tz_localize(None)
        rows = df[(df["DATETIME"] >= start) & (df["DATETIME"] <= end)].copy()
        label = f"{start:%Y-%m-%d}_to_{end:%Y-%m-%d}"
    else:
        raise ValueError(f"Unsupported PRICE_TIME_MODE / GRID_TIME_MODE: {PRICE_TIME_MODE!r}")

    if rows.empty:
        raise ValueError("No price run rows selected. Check CSV_PATH and configured date range.")
    return rows.reset_index(drop=True), label


run_rows, target_label = load_price_run_rows()
run_rows["DATETIME"] = pd.to_datetime(run_rows["DATETIME"]).dt.tz_localize(None)

print(f"Selected rows: {len(run_rows)} ({target_label})")
print("Start:", run_rows["DATETIME"].min())
print("End:  ", run_rows["DATETIME"].max())
run_rows.head()


RuntimeError: PRICE notebook could not find DASHBOARD_GRID_RUN_ROWS for GRID_DATA_SOURCE='carbon_api'. Run 1.dashboard_lca_adaptive.ipynb with RUN_GRID_NOTEBOOK_FROM_DASHBOARD=True, or run 3.1.custom_grid_carbon_intensity_api.ipynb first in the same kernel.

## Fetch Elexon Market Index half-hourly electricity prices

This uses Elexon BMRS Market Index Data. `APXMIDP` is used by default because it is the better main-case settlement-period price series for this work. `N2EXMIDP` is retained as a fallback/sensitivity provider.


In [ ]:
def _iso_z(ts) -> str:
    """Format a pandas timestamp for Elexon API calls. Naive values are treated as UTC-like."""
    ts = pd.Timestamp(ts)
    if ts.tzinfo is not None:
        ts = ts.tz_convert("UTC").tz_localize(None)
    return ts.strftime("%Y-%m-%dT%H:%MZ")


def fetch_elexon_market_index_chunk(start, end, provider: str) -> pd.DataFrame:
    url = f"{ELEXON_API_BASE.rstrip('/')}/balancing/pricing/market-index"
    params = {
        "from": _iso_z(start),
        "to": _iso_z(end),
        "dataProviders": provider,
        "format": "json",
    }
    r = requests.get(url, params=params, timeout=ELEXON_TIMEOUT_S)
    r.raise_for_status()
    payload = r.json()
    records = payload.get("data", payload.get("items", payload if isinstance(payload, list) else []))
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    df["requested_provider"] = provider
    return df


def fetch_elexon_market_index(start, end, provider: str, chunk_days: int = 7) -> pd.DataFrame:
    """Fetch Elexon Market Index Data in chunks to avoid large-query failures."""
    start = pd.Timestamp(start).tz_localize(None)
    # Add one settlement period so an inclusive configured end is captured.
    end = pd.Timestamp(end).tz_localize(None) + pd.Timedelta(minutes=30)

    chunks = []
    cursor = start
    while cursor <= end:
        chunk_end = min(cursor + pd.Timedelta(days=chunk_days), end)
        print(f"Fetching {provider}: {cursor} -> {chunk_end}")
        chunk = fetch_elexon_market_index_chunk(cursor, chunk_end, provider)
        if not chunk.empty:
            chunks.append(chunk)
        cursor = chunk_end + pd.Timedelta(minutes=30)
        time.sleep(ELEXON_REQUEST_SLEEP_S)

    if not chunks:
        return pd.DataFrame()
    return pd.concat(chunks, ignore_index=True).drop_duplicates()


def _settlement_period_to_datetime(date_series, sp_series) -> pd.Series:
    base = pd.to_datetime(date_series, errors="coerce").dt.tz_localize(None)
    sp = pd.to_numeric(sp_series, errors="coerce")
    return base + pd.to_timedelta((sp - 1) * 30, unit="m")


def normalise_elexon_market_index(raw: pd.DataFrame) -> pd.DataFrame:
    """Convert the raw Elexon response to datetime, provider, price and volume columns."""
    if raw.empty:
        return pd.DataFrame(columns=["DATETIME", "provider", "price_GBP_per_MWh", "volume_MWh"])

    df = raw.copy()
    lower_map = {c.lower(): c for c in df.columns}

    # Provider column.
    provider_col = None
    for candidate in ["dataProvider", "data_provider", "provider", "requested_provider"]:
        if candidate in df.columns:
            provider_col = candidate
            break
        if candidate.lower() in lower_map:
            provider_col = lower_map[candidate.lower()]
            break
    df["provider"] = df[provider_col] if provider_col else ELEXON_MARKET_INDEX_PROVIDER

    # Price column.
    price_col = None
    for candidate in ["price", "marketIndexPrice", "market_index_price", "value"]:
        if candidate in df.columns:
            price_col = candidate
            break
        if candidate.lower() in lower_map:
            price_col = lower_map[candidate.lower()]
            break
    if price_col is None:
        raise ValueError(f"Could not identify Elexon price column. Columns: {list(df.columns)}")
    df["price_GBP_per_MWh"] = pd.to_numeric(df[price_col], errors="coerce")

    # Volume column.
    volume_col = None
    for candidate in ["volume", "marketIndexVolume", "market_index_volume"]:
        if candidate in df.columns:
            volume_col = candidate
            break
        if candidate.lower() in lower_map:
            volume_col = lower_map[candidate.lower()]
            break
    df["volume_MWh"] = pd.to_numeric(df[volume_col], errors="coerce") if volume_col else np.nan

    # Datetime column: prefer explicit startTime, otherwise derive from settlementDate + settlementPeriod.
    dt_col = None
    for candidate in ["startTime", "start_time", "datetime", "time", "timestamp"]:
        if candidate in df.columns:
            dt_col = candidate
            break
        if candidate.lower() in lower_map:
            dt_col = lower_map[candidate.lower()]
            break

    if dt_col is not None:
        dt = pd.to_datetime(df[dt_col], errors="coerce", utc=True).dt.tz_convert(None)
    else:
        settlement_date_col = None
        settlement_period_col = None
        for candidate in ["settlementDate", "settlement_date"]:
            if candidate in df.columns:
                settlement_date_col = candidate
                break
            if candidate.lower() in lower_map:
                settlement_date_col = lower_map[candidate.lower()]
                break
        for candidate in ["settlementPeriod", "settlement_period"]:
            if candidate in df.columns:
                settlement_period_col = candidate
                break
            if candidate.lower() in lower_map:
                settlement_period_col = lower_map[candidate.lower()]
                break
        if settlement_date_col is None or settlement_period_col is None:
            raise ValueError(f"Could not identify Elexon time columns. Columns: {list(df.columns)}")
        dt = _settlement_period_to_datetime(df[settlement_date_col], df[settlement_period_col])

    df["DATETIME"] = dt.dt.floor("30min")
    df = df.dropna(subset=["DATETIME", "price_GBP_per_MWh"])
    df = df[["DATETIME", "provider", "price_GBP_per_MWh", "volume_MWh"]].copy()
    df = df.sort_values(["DATETIME", "provider"]).drop_duplicates(["DATETIME", "provider"], keep="last")
    return df.reset_index(drop=True)


price_start = run_rows["DATETIME"].min()
price_end = run_rows["DATETIME"].max()

raw_prices = fetch_elexon_market_index(price_start, price_end, ELEXON_MARKET_INDEX_PROVIDER, ELEXON_CHUNK_DAYS)
if raw_prices.empty and ELEXON_MARKET_INDEX_FALLBACK_PROVIDER:
    print(f"No records from {ELEXON_MARKET_INDEX_PROVIDER}; retrying {ELEXON_MARKET_INDEX_FALLBACK_PROVIDER}.")
    raw_prices = fetch_elexon_market_index(price_start, price_end, ELEXON_MARKET_INDEX_FALLBACK_PROVIDER, ELEXON_CHUNK_DAYS)

price_df = normalise_elexon_market_index(raw_prices)

print("Raw price rows:", len(raw_prices))
print("Clean price rows:", len(price_df))
price_df.head()


## Align prices to LCA slices

This creates the key file for the optimisation model: one row per selected time slice, with a wholesale Elexon price and an optional delivered-price uplift.


In [ ]:
def align_prices_to_run_rows(run_rows: pd.DataFrame, price_df: pd.DataFrame) -> pd.DataFrame:
    rows = run_rows.copy()
    rows["DATETIME"] = pd.to_datetime(rows["DATETIME"]).dt.tz_localize(None).dt.floor("30min")

    if price_df.empty:
        raise ValueError("No Elexon price data available to align.")

    prices = price_df.copy()
    prices["DATETIME"] = pd.to_datetime(prices["DATETIME"]).dt.tz_localize(None).dt.floor("30min")

    # If multiple providers are present, prefer the configured provider, then fallback.
    provider_order = [ELEXON_MARKET_INDEX_PROVIDER]
    if ELEXON_MARKET_INDEX_FALLBACK_PROVIDER not in provider_order:
        provider_order.append(ELEXON_MARKET_INDEX_FALLBACK_PROVIDER)

    selected = []
    for _, group in prices.groupby("DATETIME"):
        picked = None
        for p in provider_order:
            g = group[group["provider"].astype(str).str.upper() == str(p).upper()]
            if not g.empty:
                picked = g.iloc[-1]
                break
        if picked is None:
            picked = group.iloc[-1]
        selected.append(picked)

    prices_one = pd.DataFrame(selected).sort_values("DATETIME")

    aligned = rows.merge(prices_one, on="DATETIME", how="left")
    missing = aligned["price_GBP_per_MWh"].isna().sum()

    if missing:
        print(f"Warning: {missing} rows did not exact-match Elexon prices. Trying nearest 30-minute match.")
        # Nearest match, tolerance 31 min, preserving original rows.
        base_cols = [c for c in aligned.columns if c not in {"provider", "price_GBP_per_MWh", "volume_MWh"}]
        aligned_base = aligned[base_cols].copy().sort_values("DATETIME")
        nearest = pd.merge_asof(
            aligned_base,
            prices_one.sort_values("DATETIME"),
            on="DATETIME",
            direction="nearest",
            tolerance=pd.Timedelta(minutes=31),
        )
        aligned = nearest
        missing = aligned["price_GBP_per_MWh"].isna().sum()
        if missing:
            print(f"Still missing prices for {missing} rows. These remain NaN.")

    aligned["wholesale_price_GBP_per_MWh"] = aligned["price_GBP_per_MWh"]
    aligned["delivered_uplift_GBP_per_MWh"] = ELECTRICITY_DELIVERED_UPLIFT_GBP_PER_MWH
    aligned["grid_purchase_price_GBP_per_MWh"] = aligned["wholesale_price_GBP_per_MWh"] + aligned["delivered_uplift_GBP_per_MWh"]
    aligned["grid_purchase_price_GBP_per_kWh"] = aligned["grid_purchase_price_GBP_per_MWh"] / 1000.0
    aligned["price_source"] = "Elexon BMRS Market Index Data"
    aligned["price_provider_selected"] = aligned["provider"]

    return aligned


price_timeslices = align_prices_to_run_rows(run_rows, price_df)

summary_cols = [
    "DATETIME", "wholesale_price_GBP_per_MWh", "delivered_uplift_GBP_per_MWh",
    "grid_purchase_price_GBP_per_MWh", "price_provider_selected"
]
print(price_timeslices[summary_cols].describe(include="all"))
price_timeslices[summary_cols].head(12)


## Static UK onshore wind cost assumptions

These values are from the DESNZ/Arup **Renewable Energy Generation Cost and Technical Assumptions 2024 – Onshore Wind and Solar PV** report, onshore wind summary table. Units are real 2023 prices.


In [ ]:
# UK onshore wind cost assumptions from DESNZ/Arup 2024 onshore wind report.
# Real 2023 prices. Low/central/high are deliberately retained as sensitivity cases.

wind_costs = pd.DataFrame([
    {
        "asset": "onshore_wind",
        "parameter": "predevelopment_capex",
        "unit": "GBP_per_kW",
        "low": 36,
        "central": 81,
        "high": 199,
        "source": "DESNZ/Arup 2024 onshore wind and solar PV cost report",
        "notes": "Pre-development expenditure; real 2023 prices.",
    },
    {
        "asset": "onshore_wind",
        "parameter": "construction_capex",
        "unit": "GBP_per_kW",
        "low": 963,
        "central": 1204,
        "high": 1603,
        "source": "DESNZ/Arup 2024 onshore wind and solar PV cost report",
        "notes": "Capital costs during construction; real 2023 prices.",
    },
    {
        "asset": "onshore_wind",
        "parameter": "infrastructure_capex",
        "unit": "GBP_per_kW",
        "low": 242,
        "central": 303,
        "high": 403,
        "source": "DESNZ/Arup 2024 onshore wind and solar PV cost report",
        "notes": "Grid connection, substations and associated infrastructure; real 2023 prices.",
    },
    {
        "asset": "onshore_wind",
        "parameter": "total_capex",
        "unit": "GBP_per_kW",
        "low": 1241,
        "central": 1588,
        "high": 2205,
        "source": "DESNZ/Arup 2024 onshore wind and solar PV cost report",
        "notes": "Pre-development + construction + infrastructure; real 2023 prices.",
    },
    {
        "asset": "onshore_wind",
        "parameter": "om",
        "unit": "kGBP_per_MW_year",
        "low": 14.8,
        "central": 19.5,
        "high": 22.8,
        "source": "DESNZ/Arup 2024 onshore wind and solar PV cost report",
        "notes": "O&M costs; real 2023 prices.",
    },
    {
        "asset": "onshore_wind",
        "parameter": "insurance",
        "unit": "kGBP_per_MW_year",
        "low": 2.2,
        "central": 3.6,
        "high": 4.4,
        "source": "DESNZ/Arup 2024 onshore wind and solar PV cost report",
        "notes": "Insurance costs; real 2023 prices.",
    },
    {
        "asset": "onshore_wind",
        "parameter": "connection_and_uos",
        "unit": "kGBP_per_MW_year",
        "low": 0.4,
        "central": 17.0,
        "high": 45.3,
        "source": "DESNZ/Arup 2024 onshore wind and solar PV cost report",
        "notes": "Connection and Use of System charges; real 2023 prices.",
    },
    {
        "asset": "onshore_wind",
        "parameter": "total_opex",
        "unit": "kGBP_per_MW_year",
        "low": 17.4,
        "central": 40.1,
        "high": 72.4,
        "source": "DESNZ/Arup 2024 onshore wind and solar PV cost report",
        "notes": "O&M + insurance + connection/UoS; real 2023 prices.",
    },
    {
        "asset": "onshore_wind",
        "parameter": "net_load_factor",
        "unit": "fraction",
        "low": 0.330,
        "central": 0.381,
        "high": 0.413,
        "source": "DESNZ/Arup 2024 onshore wind and solar PV cost report",
        "notes": "Full-life average net load factor.",
    },
    {
        "asset": "onshore_wind",
        "parameter": "operating_lifetime",
        "unit": "years",
        "low": 25,
        "central": 35,
        "high": 40,
        "source": "DESNZ/Arup 2024 onshore wind and solar PV cost report",
        "notes": "Expected operating lifetime.",
    },
    {
        "asset": "onshore_wind",
        "parameter": "hurdle_rate",
        "unit": "fraction",
        "low": 0.058,
        "central": 0.058,
        "high": 0.058,
        "source": "DESNZ/Arup 2024 onshore wind and solar PV cost report",
        "notes": "Pre-tax real hurdle rate used in the report.",
    },
])

wind_costs["selected_case"] = WIND_COST_CASE
wind_costs["selected_value"] = wind_costs[WIND_COST_CASE]
wind_costs


## Fetch NESO electrolyser CAPEX/OPEX assumptions, otherwise use a proxy

This first tries to read the NESO Green Hydrogen Data Portal resource for PEM and Alkaline electrolyser systems. If the API fails or returns no records, the notebook falls back to a transparent low/central/high proxy table for Alkaline, PEM and SOEC.

The proxy is intended for LCOH/optimisation sensitivity analysis. It should be described as a scenario assumption, not as observed project pricing.


In [ ]:
def empty_electrolyser_cost_schema() -> pd.DataFrame:
    """Stable electrolyser-cost output schema used by NESO and proxy paths."""
    return pd.DataFrame(columns=[
        "asset", "technology", "scenario", "build_year", "capex_GBP_per_kWe",
        "efficiency_pct", "capex_GBP_per_kW_H2_HHV", "fixed_opex_pct_capex",
        "fixed_opex_GBP_per_kW_H2_HHV_year", "variable_opex_GBP_per_kWh_H2_HHV",
        "plant_lifetime_years", "source", "is_proxy", "notes",
    ])


def fetch_neso_datastore(resource_id: str, limit: int = 5000) -> pd.DataFrame:
    url = f"{NESO_API_BASE.rstrip('/')}/datastore_search"
    params = {"resource_id": resource_id, "limit": limit}
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    payload = r.json()
    if not payload.get("success", False):
        raise RuntimeError(f"NESO API returned success=False: {payload}")
    records = payload.get("result", {}).get("records", [])
    return pd.DataFrame(records)


def normalise_neso_electrolyser_costs(df: pd.DataFrame) -> pd.DataFrame:
    """Return a clean electrolyser cost table with stable column names."""
    if df.empty:
        return empty_electrolyser_cost_schema()

    out = df.copy()
    # Strip CKAN bookkeeping columns if present.
    out = out[[c for c in out.columns if not str(c).startswith("_")]].copy()

    # Do a light-touch rename only for known fields.
    rename = {
        "Tech": "technology",
        "Technology": "technology",
        "Scenario": "scenario",
        "Build Year": "build_year",
        "Build year": "build_year",
        "Capex (£/kWe)": "capex_GBP_per_kWe",
        "CAPEX (£/kWe)": "capex_GBP_per_kWe",
        "Efficiency": "efficiency_pct",
        "Efficiency (%)": "efficiency_pct",
        "Capex (£/kW H2 HHV)": "capex_GBP_per_kW_H2_HHV",
        "CAPEX (£/kW H2 HHV)": "capex_GBP_per_kW_H2_HHV",
        "Fixed Opex % of capex": "fixed_opex_pct_capex",
        "Fixed OPEX % of CAPEX": "fixed_opex_pct_capex",
        "Fixed Opex": "fixed_opex_GBP_per_kW_H2_HHV_year",
        "Fixed OPEX": "fixed_opex_GBP_per_kW_H2_HHV_year",
        "Variable Opex": "variable_opex_GBP_per_kWh_H2_HHV",
        "Variable OPEX": "variable_opex_GBP_per_kWh_H2_HHV",
        "Plant Lifetime": "plant_lifetime_years",
        "Plant lifetime": "plant_lifetime_years",
    }
    out = out.rename(columns={k: v for k, v in rename.items() if k in out.columns})

    for c in [
        "build_year", "capex_GBP_per_kWe", "efficiency_pct", "capex_GBP_per_kW_H2_HHV",
        "fixed_opex_pct_capex", "fixed_opex_GBP_per_kW_H2_HHV_year",
        "variable_opex_GBP_per_kWh_H2_HHV", "plant_lifetime_years"
    ]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    # Some sources express efficiency as a fraction rather than percent.
    if "efficiency_pct" in out.columns:
        mask = out["efficiency_pct"].between(0, 1, inclusive="both")
        out.loc[mask, "efficiency_pct"] = out.loc[mask, "efficiency_pct"] * 100.0

    # Some sources express fixed OPEX percent as 3 rather than 0.03.
    if "fixed_opex_pct_capex" in out.columns:
        mask = out["fixed_opex_pct_capex"] > 1
        out.loc[mask, "fixed_opex_pct_capex"] = out.loc[mask, "fixed_opex_pct_capex"] / 100.0

    out["asset"] = "electrolyser"
    out["source"] = "NESO Green Hydrogen Data Portal"
    out["is_proxy"] = False
    out["notes"] = "PEM and Alkaline cost assumptions from NESO resource where available."

    # Fill derived columns when possible.
    if "capex_GBP_per_kW_H2_HHV" not in out.columns or out["capex_GBP_per_kW_H2_HHV"].isna().all():
        eff = out.get("efficiency_pct", np.nan) / 100.0
        out["capex_GBP_per_kW_H2_HHV"] = out.get("capex_GBP_per_kWe", np.nan) / eff
    if "fixed_opex_GBP_per_kW_H2_HHV_year" not in out.columns or out["fixed_opex_GBP_per_kW_H2_HHV_year"].isna().all():
        out["fixed_opex_GBP_per_kW_H2_HHV_year"] = (
            out.get("capex_GBP_per_kW_H2_HHV", np.nan) * out.get("fixed_opex_pct_capex", np.nan)
        )

    # Ensure stable column order, retaining any extra API columns at the end.
    stable = list(empty_electrolyser_cost_schema().columns)
    for c in stable:
        if c not in out.columns:
            out[c] = np.nan
    extras = [c for c in out.columns if c not in stable]
    return out[stable + extras]


def proxy_electrolyser_costs(build_year: int = PROXY_ELECTROLYSER_BUILD_YEAR) -> pd.DataFrame:
    """Transparent fallback electrolyser CAPEX/OPEX scenarios for optimisation.

    Values are intended as modelling assumptions. They keep the LCOH workflow
    runnable when the NESO data resource is unavailable and should be sensitivity
    tested rather than treated as observed UK project prices.
    """
    rows = []
    scenarios = ["low", "central", "high"]

    for technology, capex_cases in PROXY_ELECTROLYSER_CAPEX_GBP_PER_KWE.items():
        efficiency_pct = float(PROXY_ELECTROLYSER_EFFICIENCY_PCT.get(technology, np.nan))
        fixed_opex_pct = float(PROXY_ELECTROLYSER_FIXED_OPEX_PCT_CAPEX.get(technology, np.nan))
        lifetime_years = float(PROXY_ELECTROLYSER_LIFETIME_YEARS.get(technology, np.nan))
        efficiency_fraction = efficiency_pct / 100.0 if efficiency_pct and not pd.isna(efficiency_pct) else np.nan

        for scenario in scenarios:
            capex_kwe = capex_cases.get(scenario, np.nan) if isinstance(capex_cases, dict) else np.nan
            capex_kw_h2_hhv = capex_kwe / efficiency_fraction if efficiency_fraction else np.nan
            fixed_opex_kw_h2_year = capex_kw_h2_hhv * fixed_opex_pct if not pd.isna(capex_kw_h2_hhv) else np.nan

            rows.append({
                "asset": "electrolyser",
                "technology": technology,
                "scenario": scenario,
                "build_year": build_year,
                "capex_GBP_per_kWe": capex_kwe,
                "efficiency_pct": efficiency_pct,
                "capex_GBP_per_kW_H2_HHV": capex_kw_h2_hhv,
                "fixed_opex_pct_capex": fixed_opex_pct,
                "fixed_opex_GBP_per_kW_H2_HHV_year": fixed_opex_kw_h2_year,
                "variable_opex_GBP_per_kWh_H2_HHV": 0.0,
                "plant_lifetime_years": lifetime_years,
                "source": "Proxy: literature-informed low/central/high scenario assumption",
                "is_proxy": True,
                "notes": (
                    "Fallback used because the NESO electrolyser CAPEX/OPEX resource "
                    "was unavailable or empty at run time. Values should be sensitivity-tested."
                ),
            })

    return pd.DataFrame(rows, columns=empty_electrolyser_cost_schema().columns)


try:
    neso_raw = fetch_neso_datastore(NESO_ELECTROLYSER_RESOURCE_ID, NESO_LIMIT)
except Exception as exc:
    print("NESO electrolyser API fetch failed:")
    print(repr(exc))
    neso_raw = pd.DataFrame()

neso_electrolyser_costs = normalise_neso_electrolyser_costs(neso_raw)

if neso_electrolyser_costs.empty and USE_ELECTROLYSER_PROXY_IF_NESO_EMPTY:
    print("NESO electrolyser dataset returned no records. Using proxy electrolyser CAPEX/OPEX assumptions.")
    neso_electrolyser_costs = proxy_electrolyser_costs()
elif neso_electrolyser_costs.empty:
    print("NESO electrolyser dataset returned no records and proxy fallback is disabled.")
else:
    print("NESO electrolyser rows:", len(neso_electrolyser_costs))

print("Electrolyser assumption rows:", len(neso_electrolyser_costs))
print("Using proxy rows:", bool(neso_electrolyser_costs.get("is_proxy", pd.Series(dtype=bool)).fillna(False).any()))
neso_electrolyser_costs.head(12)


## Optional battery cost placeholders

There is no battery-cost API wired in here. The notebook writes a clear placeholder table so the optimiser has a stable input schema, but the values remain `NaN` unless you set them in `dashboard_config.py`.


In [ ]:
def _dict_case_value(d, case):
    if isinstance(d, dict):
        return d.get(case)
    return np.nan

battery_costs = pd.DataFrame([
    {
        "asset": "battery",
        "parameter": "battery_capex",
        "unit": "GBP_per_MWh",
        "low": _dict_case_value(BATTERY_CAPEX_GBP_PER_MWH, "low"),
        "central": _dict_case_value(BATTERY_CAPEX_GBP_PER_MWH, "central"),
        "high": _dict_case_value(BATTERY_CAPEX_GBP_PER_MWH, "high"),
        "source": "user_config_required",
        "notes": "Set BATTERY_CAPEX_GBP_PER_MWH in dashboard_config.py once a source is chosen.",
    },
    {
        "asset": "battery",
        "parameter": "battery_fixed_opex_pct_capex",
        "unit": "fraction_of_capex_per_year",
        "low": _dict_case_value(BATTERY_FIXED_OPEX_PCT_CAPEX, "low"),
        "central": _dict_case_value(BATTERY_FIXED_OPEX_PCT_CAPEX, "central"),
        "high": _dict_case_value(BATTERY_FIXED_OPEX_PCT_CAPEX, "high"),
        "source": "user_config_required",
        "notes": "Set BATTERY_FIXED_OPEX_PCT_CAPEX in dashboard_config.py once a source is chosen.",
    },
    {
        "asset": "battery",
        "parameter": "battery_lifetime",
        "unit": "years",
        "low": _dict_case_value(BATTERY_LIFETIME_YEARS, "low"),
        "central": _dict_case_value(BATTERY_LIFETIME_YEARS, "central"),
        "high": _dict_case_value(BATTERY_LIFETIME_YEARS, "high"),
        "source": "user_config_required",
        "notes": "Set BATTERY_LIFETIME_YEARS in dashboard_config.py once a source is chosen.",
    },
])

battery_costs["selected_case"] = COST_CASE
battery_costs["selected_value"] = battery_costs[COST_CASE]
battery_costs


## Build optimisation-ready cost input tables


In [ ]:
def capital_recovery_factor(rate: float, lifetime_years: float) -> float:
    if pd.isna(rate) or pd.isna(lifetime_years):
        return np.nan
    rate = float(rate)
    lifetime_years = float(lifetime_years)
    if rate == 0:
        return 1.0 / lifetime_years
    return rate * (1 + rate) ** lifetime_years / ((1 + rate) ** lifetime_years - 1)


def wind_annual_cost_per_mw(case: str = "central") -> dict:
    vals = wind_costs.set_index("parameter")[case]
    capex_per_mw = float(vals["total_capex"]) * 1000.0  # GBP/kW -> GBP/MW
    total_opex_per_mw_year = float(vals["total_opex"]) * 1000.0  # kGBP/MW/a -> GBP/MW/a
    lifetime = float(vals["operating_lifetime"])
    rate = float(vals["hurdle_rate"])
    crf = capital_recovery_factor(rate, lifetime)
    return {
        "case": case,
        "wind_capex_GBP_per_MW": capex_per_mw,
        "wind_total_opex_GBP_per_MW_year": total_opex_per_mw_year,
        "wind_lifetime_years": lifetime,
        "wind_discount_rate": rate,
        "wind_crf": crf,
        "wind_annualised_capex_GBP_per_MW_year": capex_per_mw * crf,
        "wind_total_annual_cost_GBP_per_MW_year": capex_per_mw * crf + total_opex_per_mw_year,
    }

wind_annual_costs = pd.DataFrame([wind_annual_cost_per_mw(c) for c in ["low", "central", "high"]])
wind_annual_costs


In [ ]:
# Long-form assumptions table for the optimiser.
static_cost_assumptions = pd.concat([
    wind_costs,
    battery_costs,
], ignore_index=True, sort=False)

# Electrolyser assumptions are kept as a separate table because they have a wider schema
# by technology, scenario and build year. If NESO is unavailable this table contains proxy rows.
print("Static wind/battery assumption rows:", len(static_cost_assumptions))
print("Electrolyser assumption rows:       ", len(neso_electrolyser_costs))
static_cost_assumptions.head(20)


## Save outputs

Files are written to `PRICE_OUTPUT_DIR` using the same target label as the other notebooks.


In [ ]:
safe_label = str(target_label).replace(" ", "_").replace(":", "-").replace("/", "-")

raw_price_path = PRICE_OUTPUT_DIR / f"elexon_market_index_raw_{safe_label}.csv"
clean_price_path = PRICE_OUTPUT_DIR / f"elexon_market_index_clean_{safe_label}.csv"
price_timeslices_path = PRICE_OUTPUT_DIR / f"price_timeslices_{safe_label}.csv"
static_costs_path = PRICE_OUTPUT_DIR / "static_cost_assumptions.csv"
wind_annual_costs_path = PRICE_OUTPUT_DIR / "wind_annual_costs_per_MW.csv"
neso_raw_path = PRICE_OUTPUT_DIR / "neso_electrolyser_costs_raw.csv"
neso_clean_path = PRICE_OUTPUT_DIR / "neso_electrolyser_costs_clean.csv"
electrolyser_clean_path = PRICE_OUTPUT_DIR / "electrolyser_costs_clean.csv"
config_snapshot_path = PRICE_OUTPUT_DIR / f"price_config_snapshot_{safe_label}.json"

raw_prices.to_csv(raw_price_path, index=False)
price_df.to_csv(clean_price_path, index=False)
price_timeslices.to_csv(price_timeslices_path, index=False)
static_cost_assumptions.to_csv(static_costs_path, index=False)
wind_annual_costs.to_csv(wind_annual_costs_path, index=False)
neso_raw.to_csv(neso_raw_path, index=False)
neso_electrolyser_costs.to_csv(neso_clean_path, index=False)
neso_electrolyser_costs.to_csv(electrolyser_clean_path, index=False)

config_snapshot = {
    "PRICE_TIME_MODE": PRICE_TIME_MODE,
    "PRICE_SINGLE_DATETIME": PRICE_SINGLE_DATETIME,
    "PRICE_RANGE_START": PRICE_RANGE_START,
    "PRICE_RANGE_END": PRICE_RANGE_END,
    "PRICE_GRID_CSV_PATH": PRICE_GRID_CSV_PATH,
    "ELEXON_API_BASE": ELEXON_API_BASE,
    "ELEXON_MARKET_INDEX_PROVIDER": ELEXON_MARKET_INDEX_PROVIDER,
    "ELEXON_MARKET_INDEX_FALLBACK_PROVIDER": ELEXON_MARKET_INDEX_FALLBACK_PROVIDER,
    "ELECTRICITY_DELIVERED_UPLIFT_GBP_PER_MWH": ELECTRICITY_DELIVERED_UPLIFT_GBP_PER_MWH,
    "NESO_ELECTROLYSER_RESOURCE_ID": NESO_ELECTROLYSER_RESOURCE_ID,
    "USE_ELECTROLYSER_PROXY_IF_NESO_EMPTY": USE_ELECTROLYSER_PROXY_IF_NESO_EMPTY,
    "PROXY_ELECTROLYSER_BUILD_YEAR": PROXY_ELECTROLYSER_BUILD_YEAR,
    "PROXY_ELECTROLYSER_CAPEX_GBP_PER_KWE": PROXY_ELECTROLYSER_CAPEX_GBP_PER_KWE,
    "PROXY_ELECTROLYSER_EFFICIENCY_PCT": PROXY_ELECTROLYSER_EFFICIENCY_PCT,
    "PROXY_ELECTROLYSER_FIXED_OPEX_PCT_CAPEX": PROXY_ELECTROLYSER_FIXED_OPEX_PCT_CAPEX,
    "PROXY_ELECTROLYSER_LIFETIME_YEARS": PROXY_ELECTROLYSER_LIFETIME_YEARS,
    "COST_CASE": COST_CASE,
    "WIND_COST_CASE": WIND_COST_CASE,
    "outputs": {
        "raw_price_path": str(raw_price_path),
        "clean_price_path": str(clean_price_path),
        "price_timeslices_path": str(price_timeslices_path),
        "static_costs_path": str(static_costs_path),
        "wind_annual_costs_path": str(wind_annual_costs_path),
        "neso_raw_path": str(neso_raw_path),
        "neso_clean_path": str(neso_clean_path),
        "electrolyser_clean_path": str(electrolyser_clean_path),
    },
}
config_snapshot_path.write_text(json.dumps(config_snapshot, indent=2), encoding="utf-8")

print("Saved:")
for p in [
    raw_price_path, clean_price_path, price_timeslices_path, static_costs_path,
    wind_annual_costs_path, neso_raw_path, neso_clean_path, electrolyser_clean_path, config_snapshot_path,
]:
    print(" -", p)


## Quick use in the optimisation notebook

The optimisation notebook should read:

```python
price_slices = pd.read_csv("price_outputs/price_timeslices_<label>.csv", parse_dates=["DATETIME"])
wind_costs = pd.read_csv("price_outputs/wind_annual_costs_per_MW.csv")
static_costs = pd.read_csv("price_outputs/static_cost_assumptions.csv")
electrolyser_costs = pd.read_csv("price_outputs/electrolyser_costs_clean.csv")
```

`electrolyser_costs_clean.csv` contains NESO rows if the NESO resource is available; otherwise it contains the proxy rows and `is_proxy == True`.

Then the variable grid-electricity term is:

```python
grid_cost_t = grid_import_MWh_t * grid_purchase_price_GBP_per_MWh_t
```

and the annualised wind term is:

```python
wind_cost = wind_capacity_MW * wind_total_annual_cost_GBP_per_MW_year
```

For a 1 MW electrolyser, the central electrolyser CAPEX term is:

```python
electrolyser_capex = electrolyser_capacity_kW * capex_GBP_per_kWe
```

and the fixed OPEX term is:

```python
electrolyser_fixed_opex_year = electrolyser_capex * fixed_opex_pct_capex
```
